In [1]:
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import sklearn
import imblearn
import itertools
import sys
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn import metrics as metrics
import warnings
warnings.filterwarnings('ignore')

In [2]:
test = pd.read_csv("UNSW_NB15_testing-set.csv")
test = test.iloc[:,:-1] 

In [3]:
train = pd.read_csv("UNSW_NB15_training-set.csv")
train = train.iloc[:,:-1] 

In [4]:
y = train["attack_cat"].values
from collections import Counter
Counter(y)

Counter({'Normal': 37000,
         'Reconnaissance': 3496,
         'Backdoor': 583,
         'DoS': 4089,
         'Exploits': 11132,
         'Analysis': 677,
         'Fuzzers': 6062,
         'Worms': 44,
         'Shellcode': 378,
         'Generic': 18871})

In [5]:
y1 = test["attack_cat"].values
from collections import Counter
Counter(y1)

Counter({'Normal': 56000,
         'Backdoor': 1746,
         'Analysis': 2000,
         'Fuzzers': 18184,
         'Shellcode': 1133,
         'Reconnaissance': 10491,
         'Exploits': 33393,
         'DoS': 12264,
         'Worms': 130,
         'Generic': 40000})

In [6]:
from sklearn.preprocessing import LabelEncoder
encodings = dict()
for c in train.columns:
    
    if train[c].dtype == "object":
        encodings[c] = LabelEncoder()
        encodings[c]
        train[c] = encodings[c].fit_transform(train[c])

In [7]:
y = train.pop("attack_cat").values
X = train.values

In [8]:
train.dtypes

id                     int64
dur                  float64
proto                  int32
service                int32
state                  int32
spkts                  int64
dpkts                  int64
sbytes                 int64
dbytes                 int64
rate                 float64
sttl                   int64
dttl                   int64
sload                float64
dload                float64
sloss                  int64
dloss                  int64
sinpkt               float64
dinpkt               float64
sjit                 float64
djit                 float64
swin                   int64
stcpb                  int64
dtcpb                  int64
dwin                   int64
tcprtt               float64
synack               float64
ackdat               float64
smean                  int64
dmean                  int64
trans_depth            int64
response_body_len      int64
ct_srv_src             int64
ct_state_ttl           int64
ct_dst_ltm             int64
ct_src_dport_l

In [9]:
from sklearn.preprocessing import StandardScaler
X = StandardScaler().fit_transform(X)

In [10]:
!pip install mlxtend

In [11]:
from mlxtend.feature_selection import SequentialFeatureSelector as sfs


In [12]:
clf = RandomForestClassifier(n_estimators=100, n_jobs=-1)

sfs1 = sfs(clf,
           k_features=30,
           forward=True,
           floating=False,
           verbose=2,
           scoring='accuracy',
           cv=5)

sfs1 = sfs1.fit(X, y)

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    9.7s remaining:    0.0s
[Parallel(n_jobs=1)]: Done  43 out of  43 | elapsed:  3.0min finished

[2022-03-04 17:54:26] Features: 1/30 -- score: 0.7344886446294898[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    5.8s remaining:    0.0s
[Parallel(n_jobs=1)]: Done  42 out of  42 | elapsed:  3.4min finished

[2022-03-04 17:57:49] Features: 2/30 -- score: 0.783254859265255[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    5.6s remaining:    0.0s
[Parallel(n_jobs=1)]: Done  41 out of  41 | elapsed:  3.4min finished

[2022-03-04 18:01:14] Features: 3/30 -- score: 0.7949999148428073[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   

In [13]:
X_train_sfs, X_test_sfs, y_train_sfs, y_test_sfs = train_test_split(X[:, list(sfs1.k_feature_idx_)], y, test_size=0.2, random_state=42)

In [14]:
#KNN
from sklearn.neighbors import KNeighborsClassifier

neigh = KNeighborsClassifier(n_neighbors=5,weights='uniform')
neigh.fit(X_train_sfs, y_train_sfs)


KNeighborsClassifier()

In [15]:
y_pred_sfs = neigh.predict(X_test_sfs)
print(Counter(y_pred_sfs))
print(Counter(y_test_sfs))

Counter({6: 7976, 5: 3616, 3: 2324, 4: 880, 2: 872, 7: 574, 0: 135, 1: 67, 8: 23})
Counter({6: 7418, 5: 3723, 3: 2275, 4: 1212, 2: 786, 7: 723, 0: 131, 1: 117, 8: 75, 9: 7})


In [16]:
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
 
results = confusion_matrix(y_test_sfs, y_pred_sfs) 
print('Confusion Matrix :')
print(results) 
print('Accuracy Score :',accuracy_score(y_test_sfs, y_pred_sfs))
print('Report : ')
print(classification_report(y_test_sfs, y_pred_sfs))

Confusion Matrix :
[[  14    8   30   49   29    0    1    0    0    0]
 [  14    2   20   47   26    1    5    2    0    0]
 [  17    9  328  332   34    6   45   13    2    0]
 [  50   26  334 1527   89    3  167   76    3    0]
 [  37   15   59  125  494    1  465   15    1    0]
 [   0    0   26   66    6 3602   19    3    1    0]
 [   2    1   23   47  178    0 7124   41    2    0]
 [   1    6   48  122   23    1  124  397    1    0]
 [   0    0    4    4    1    2   25   26   13    0]
 [   0    0    0    5    0    0    1    1    0    0]]
Accuracy Score : 0.8198821886196636
Report : 
              precision    recall  f1-score   support

           0       0.10      0.11      0.11       131
           1       0.03      0.02      0.02       117
           2       0.38      0.42      0.40       786
           3       0.66      0.67      0.66      2275
           4       0.56      0.41      0.47      1212
           5       1.00      0.97      0.98      3723
           6       0.89  

In [17]:
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10)
predicted = cross_val_predict(neigh, X_train_sfs, y_train_sfs, cv=skf)
print('Accuracy Score :',accuracy_score(y_train_sfs, predicted))
print('Report : ')
print(classification_report(y_train_sfs, predicted))

Accuracy Score : 0.8164730888939498
Report : 
              precision    recall  f1-score   support

           0       0.06      0.06      0.06       546
           1       0.03      0.02      0.02       466
           2       0.38      0.41      0.39      3303
           3       0.64      0.65      0.65      8857
           4       0.55      0.41      0.47      4850
           5       0.99      0.97      0.98     15148
           6       0.89      0.96      0.93     29582
           7       0.68      0.53      0.60      2773
           8       0.52      0.13      0.21       303
           9       0.00      0.00      0.00        37

    accuracy                           0.82     65865
   macro avg       0.47      0.41      0.43     65865
weighted avg       0.81      0.82      0.81     65865



In [18]:
#SVM
from sklearn.svm import SVC

clf = SVC(gamma='auto',decision_function_shape='ovo')
clf.fit(X_train_sfs, y_train_sfs)

SVC(decision_function_shape='ovo', gamma='auto')

In [19]:
y_pred_sfs=clf.predict(X_test_sfs)
print(Counter(y_pred_sfs))
print(Counter(y_test_sfs))

Counter({6: 8213, 5: 3603, 3: 2498, 4: 1098, 7: 686, 2: 369})
Counter({6: 7418, 5: 3723, 3: 2275, 4: 1212, 2: 786, 7: 723, 0: 131, 1: 117, 8: 75, 9: 7})


In [20]:
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
 
results = confusion_matrix(y_test_sfs, y_pred_sfs) 
print('Confusion Matrix :')
print(results) 
print('Accuracy Score :',accuracy_score(y_test_sfs, y_pred_sfs))
print('Report : ')
print(classification_report(y_test_sfs, y_pred_sfs))

Confusion Matrix :
[[   0    0   13   51   59    0    8    0    0    0]
 [   0    0    3   39   64    0    8    3    0    0]
 [   0    0  159  379   70    3  121   54    0    0]
 [   0    0  135 1650  169    1  243   77    0    0]
 [   0    0   19  108  647    3  409   26    0    0]
 [   0    0    2   81    8 3593   28   11    0    0]
 [   0    0   17   36   48    0 7200  117    0    0]
 [   0    0   21  138   30    3  165  366    0    0]
 [   0    0    0   11    3    0   29   32    0    0]
 [   0    0    0    5    0    0    2    0    0    0]]
Accuracy Score : 0.8268051254023198
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.43      0.20      0.28       786
           3       0.66      0.73      0.69      2275
           4       0.59      0.53      0.56      1212
           5       1.00      0.97      0.98      3723
           6       0.88  

In [21]:
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10)
predicted = cross_val_predict(clf, X_train_sfs, y_train_sfs, cv=skf)
print('Accuracy Score :',accuracy_score(y_train_sfs, predicted))
print('Report : ')
print(classification_report(y_train_sfs, predicted))

Accuracy Score : 0.8248538677598117
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.42      0.20      0.27      3303
           3       0.64      0.71      0.68      8857
           4       0.60      0.53      0.56      4850
           5       1.00      0.96      0.98     15148
           6       0.88      0.97      0.92     29582
           7       0.53      0.52      0.52      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.82     65865
   macro avg       0.41      0.39      0.39     65865
weighted avg       0.80      0.82      0.81     65865



In [22]:
#Logistic Regression
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(random_state=0,solver='saga',multi_class='multinomial').fit(X_train_sfs, y_train_sfs)

In [23]:
y_pred_sfs = clf.predict(X_test_sfs)
print(Counter(y_pred_sfs))
print(Counter(y_test_sfs))

Counter({6: 9439, 5: 3926, 3: 1525, 7: 698, 4: 595, 2: 284})
Counter({6: 7418, 5: 3723, 3: 2275, 4: 1212, 2: 786, 7: 723, 0: 131, 1: 117, 8: 75, 9: 7})


In [24]:
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
 
print('Accuracy Score :',accuracy_score(y_test_sfs, y_pred_sfs))
print('Report : ')
print(classification_report(y_test_sfs, y_pred_sfs))

Accuracy Score : 0.7387502277281837
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.40      0.15      0.21       786
           3       0.56      0.38      0.45      2275
           4       0.54      0.26      0.35      1212
           5       0.92      0.97      0.94      3723
           6       0.73      0.93      0.82      7418
           7       0.49      0.48      0.48       723
           8       0.00      0.00      0.00        75
           9       0.00      0.00      0.00         7

    accuracy                           0.74     16467
   macro avg       0.36      0.32      0.33     16467
weighted avg       0.70      0.74      0.70     16467



In [25]:
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10)
predicted = cross_val_predict(clf, X_train_sfs, y_train_sfs, cv=skf)
print('Accuracy Score :',accuracy_score(y_train_sfs, predicted))
print('Report : ')
print(classification_report(y_train_sfs, predicted))

Accuracy Score : 0.7418963030441054
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.39      0.14      0.21      3303
           3       0.57      0.39      0.46      8857
           4       0.54      0.26      0.35      4850
           5       0.91      0.97      0.94     15148
           6       0.74      0.94      0.83     29582
           7       0.47      0.47      0.47      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.74     65865
   macro avg       0.36      0.32      0.33     65865
weighted avg       0.70      0.74      0.71     65865



In [26]:
#Multi Layer Perceptron
from sklearn.neural_network import MLPClassifier
clf = MLPClassifier(alpha=1)
clf.fit(X_train_sfs, y_train_sfs)
clf

MLPClassifier(alpha=1)

In [27]:
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
 
results = confusion_matrix(y_test_sfs, y_pred_sfs) 
print('Confusion Matrix :')
print(results) 
print('Accuracy Score :',accuracy_score(y_test_sfs, y_pred_sfs))
print('Report : ')
print(classification_report(y_test_sfs, y_pred_sfs))

Confusion Matrix :
[[   0    0    7   10   40   34   38    2    0    0]
 [   0    0    2   10   55   31   15    4    0    0]
 [   0    0  115  125   44   59  386   57    0    0]
 [   0    0  106  858  115   93  982  121    0    0]
 [   0    0   24   62  319   69  708   30    0    0]
 [   0    0    1   48    1 3610   53   10    0    0]
 [   0    0    9  346   16   22 6919  106    0    0]
 [   0    0   20   63    5    6  285  344    0    0]
 [   0    0    0    0    0    0   51   24    0    0]
 [   0    0    0    3    0    2    2    0    0    0]]
Accuracy Score : 0.7387502277281837
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.40      0.15      0.21       786
           3       0.56      0.38      0.45      2275
           4       0.54      0.26      0.35      1212
           5       0.92      0.97      0.94      3723
           6       0.73  

In [28]:
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10)
predicted = cross_val_predict(clf, X_train_sfs, y_train_sfs, cv=skf)
print('Accuracy Score :',accuracy_score(y_train_sfs, predicted))
print('Report : ')
print(classification_report(y_train_sfs, predicted))

Accuracy Score : 0.8095498367873681
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.40      0.14      0.20      3303
           3       0.63      0.69      0.66      8857
           4       0.56      0.42      0.48      4850
           5       0.99      0.96      0.98     15148
           6       0.84      0.97      0.90     29582
           7       0.52      0.49      0.50      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.81     65865
   macro avg       0.39      0.37      0.37     65865
weighted avg       0.78      0.81      0.79     65865



In [29]:
#Random Forest
from sklearn.ensemble import RandomForestRegressor
forest = RandomForestRegressor()
_ = forest.fit(X_train_sfs, y_train_sfs)
arr = forest.predict(X_train_sfs).astype(int)
print(classification_report(y_train_sfs, arr))
print('Accuracy Score :',accuracy_score(y_train_sfs, arr))

              precision    recall  f1-score   support

           0       1.00      0.08      0.15       546
           1       0.58      0.06      0.12       466
           2       0.32      0.93      0.47      3303
           3       0.85      0.47      0.61      8857
           4       0.84      0.71      0.77      4850
           5       0.68      0.97      0.80     15148
           6       0.97      0.77      0.86     29582
           7       0.89      0.67      0.76      2773
           8       0.09      0.00      0.01       303
           9       0.00      0.00      0.00        37

    accuracy                           0.76     65865
   macro avg       0.62      0.47      0.45     65865
weighted avg       0.84      0.76      0.77     65865

Accuracy Score : 0.7594018067258786


In [30]:
arr = forest.predict(X_test_sfs).astype(int)
print(classification_report(y_test_sfs, arr))
print('Accuracy Score :',accuracy_score(y_test_sfs, arr))

              precision    recall  f1-score   support

           0       0.91      0.08      0.14       131
           1       0.14      0.02      0.03       117
           2       0.27      0.80      0.40       786
           3       0.72      0.36      0.48      2275
           4       0.66      0.62      0.64      1212
           5       0.62      0.97      0.75      3723
           6       0.97      0.72      0.83      7418
           7       0.95      0.63      0.76       723
           8       0.00      0.00      0.00        75
           9       0.00      0.00      0.00         7

    accuracy                           0.71     16467
   macro avg       0.52      0.42      0.40     16467
weighted avg       0.79      0.71      0.71     16467

Accuracy Score : 0.7052893666120119


In [31]:
#NaiveBayes
from sklearn.naive_bayes import GaussianNB
classifier = GaussianNB()
classifier.fit(X_train_sfs, y_train_sfs)
arr = forest.predict(X_train_sfs).astype(int)
print(classification_report(y_train_sfs, arr))

              precision    recall  f1-score   support

           0       1.00      0.08      0.15       546
           1       0.58      0.06      0.12       466
           2       0.32      0.93      0.47      3303
           3       0.85      0.47      0.61      8857
           4       0.84      0.71      0.77      4850
           5       0.68      0.97      0.80     15148
           6       0.97      0.77      0.86     29582
           7       0.89      0.67      0.76      2773
           8       0.09      0.00      0.01       303
           9       0.00      0.00      0.00        37

    accuracy                           0.76     65865
   macro avg       0.62      0.47      0.45     65865
weighted avg       0.84      0.76      0.77     65865



In [32]:
arr = classifier.predict(X_test_sfs).astype(int)
print(classification_report(y_test_sfs, arr))
print('Accuracy Score :',accuracy_score(y_test_sfs, arr))

              precision    recall  f1-score   support

           0       0.07      0.90      0.13       131
           1       0.00      0.03      0.00       117
           2       0.12      0.01      0.02       786
           3       0.33      0.48      0.39      2275
           4       0.25      0.15      0.19      1212
           5       0.93      0.55      0.69      3723
           6       0.98      0.31      0.47      7418
           7       0.04      0.04      0.04       723
           8       0.02      0.99      0.05        75
           9       0.01      0.43      0.01         7

    accuracy                           0.35     16467
   macro avg       0.28      0.39      0.20     16467
weighted avg       0.73      0.35      0.44     16467

Accuracy Score : 0.35391996113439


In [33]:
#DecisionTree Entropy
from sklearn.tree import DecisionTreeClassifier
clf_entropy = DecisionTreeClassifier(criterion = "entropy", random_state = 100,max_depth = 3, min_samples_leaf = 5)
  
clf_entropy.fit(X_train_sfs, y_train_sfs)
y_pred = clf_entropy.predict(X_train_sfs)
print(classification_report(y_train_sfs, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.00      0.00      0.00      3303
           3       0.49      0.60      0.54      8857
           4       0.00      0.00      0.00      4850
           5       1.00      0.96      0.98     15148
           6       0.69      0.95      0.80     29582
           7       0.00      0.00      0.00      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.73     65865
   macro avg       0.22      0.25      0.23     65865
weighted avg       0.61      0.73      0.66     65865



In [34]:
y_pred = clf_entropy.predict(X_test_sfs)
print(classification_report(y_test_sfs, y_pred))
print('Accuracy Score :',accuracy_score(y_test_sfs, arr))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.00      0.00      0.00       786
           3       0.51      0.60      0.55      2275
           4       0.00      0.00      0.00      1212
           5       1.00      0.96      0.98      3723
           6       0.69      0.95      0.80      7418
           7       0.00      0.00      0.00       723
           8       0.00      0.00      0.00        75
           9       0.00      0.00      0.00         7

    accuracy                           0.73     16467
   macro avg       0.22      0.25      0.23     16467
weighted avg       0.61      0.73      0.66     16467

Accuracy Score : 0.35391996113439


In [35]:
#DecisionTree Gini Index
clf_gini = DecisionTreeClassifier(criterion = "gini",random_state = 100,max_depth=3, min_samples_leaf=5)
clf_gini.fit(X_train_sfs, y_train_sfs)
y_pred = clf_gini.predict(X_train_sfs)
print(classification_report(y_train_sfs, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.00      0.00      0.00      3303
           3       0.62      0.55      0.58      8857
           4       0.00      0.00      0.00      4850
           5       1.00      0.92      0.96     15148
           6       0.67      1.00      0.80     29582
           7       0.00      0.00      0.00      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.73     65865
   macro avg       0.23      0.25      0.23     65865
weighted avg       0.61      0.73      0.66     65865



In [36]:
y_pred = clf_gini.predict(X_test_sfs)
print(classification_report(y_test_sfs, y_pred))
print('Accuracy Score :',accuracy_score(y_test_sfs, arr))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.00      0.00      0.00       786
           3       0.66      0.56      0.60      2275
           4       0.00      0.00      0.00      1212
           5       1.00      0.92      0.96      3723
           6       0.67      1.00      0.80      7418
           7       0.00      0.00      0.00       723
           8       0.00      0.00      0.00        75
           9       0.00      0.00      0.00         7

    accuracy                           0.74     16467
   macro avg       0.23      0.25      0.24     16467
weighted avg       0.62      0.74      0.66     16467

Accuracy Score : 0.35391996113439
